<a href="https://colab.research.google.com/github/anandita-3217/EAI-Assignments/blob/main/EAI-Assignments%20/Search/SE26MAID034_PathFinding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Using Indian Metro cities as nodes calculating the implementing BFS, DFS, UCS, IDDFS

In [ ]:
from collections import deque
from heapq import heappush, heappop
from itertools import count


In [ ]:
Cities_Distances = [
    [0, 1419, 2121, 1499, 2107, 1453],      # Delhi
    [1419, 0, 984, 711, 1290, 1939],        # Mumbai
    [2121, 984, 0, 575, 346, 1875],         # Bengaluru
    [1499, 711, 575, 0, 633, 1492],         # Hyderabad
    [2107, 1290, 346, 633, 0, 1684],        # Chennai
    [1453, 1939, 1875, 1492, 1684, 0]       # Kolkata
]
City_Names = ["Delhi", "Mumbai", "Bengaluru", "Hyderabad", "Chennai", "Kolkata"]

In [1]:
class Problem:
  def __init__(self, src, goal=None):
    self.src = src
    self.goal = goal

  def actions(self,state):
    raise NotImplementedError

  def result(self,state, action):
    raise NotImplementedError

  def goal_test(self, state):
    if isinstance(self.goal, list):
      return state in self.goal
    else:
      return state == self.goal

  def path_cost(self, c, state1, action, state2):
    return c+1

  def value(self, state):
    raise NotImplementedError

In [6]:
class Node:
  def __init__(self, state, parent=None, action=None, path_cost=0):
    self.state = state
    self.parent = parent
    self.action = action
    self.path_cost = path_cost
    self.depth = 0
    if parent:
      self.depth = parent.depth+1

  def __repr__(self):
    return "<Node {}>".format(self.state)

  def __lt__(self,node):
    return self.state < node.state

  def expand(self, problem):
    return [self.child_node(problem, action) for action in problem.actions(self.state)]

  def child_node(self, problem, action):
    next_state = problem.result(self.state, action)
    next_node=Node(next_state, self, action, problem.path_cost(self.path_cost, self.state, action, next_state))
    return next_node

  def solution(self):
    return [node.action for node in self.path()[1:]]

  def path(self):
    node, path_back = self, []
    while node:
      path_back.append(node)
      node = node.parent
    return list(reversed(path_back))

  def path_states(self):
    return [node.state for node in self.path()]

  def __eq__(self, other):
    return isinstance(other, Node) and self.state == other.state

  def __hash__(self):
    return hash(self.state)


In [3]:
def depth_first_search(problem):
  frontier = [(Node(problem.src))]
  explored = set()
  while frontier:
    node = frontier.pop()
    if problem.goal_test(node.state):
      return node
    explored.add(node.state)
    frontier.extend(child for child in node.expand(problem) if child.state not in explored and child not in frontier)
  return None


In [4]:
def breadth_first_search(problem):
  node = Node(problem.src)
  if problem.goal_test(node.state):
    return node
  frontier = deque([node])
  explored = set()
  while frontier:
    node = frontier.popleft()
    explored.add(node.state)
    for child in node.expand(problem):
      if child.state not in explored and child not in frontier:
        if problem.goal_test(child.state):
          return child
        frontier.append(child)
  return None

In [ ]:
import heapq
def best_first_search(problem, f):
  tie = count()
  node =Node(problem.src)
  frontier = [(f(node), next(tie), node)]
  reached = {problem.src: node}
  while frontier:
    _,_, node = heapq.heappop(frontier)
    if problem.goal_test(node.state):
      return node
    for child in node.expand(problem):
      if child.state not in reached or child.path_cost < reached[child.state].path_cost:
        reached[child.state] = child
        heappush(frontier,(f(child), tie, child))
        tie += 1
  return None

In [7]:
def uniform_cost_search(problem):
  return best_first_search(problem, f = lambda node: node.path_cost)

In [8]:
def a_search(problem):
  return best_first_search(problem, f = lambda node: node.path_cost+problem.h(node))